In [1]:
!pip install adapters -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 295.5/295.5 kB 10.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 32.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 23.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.20.0 requires huggingface-hub<2.0,>=1.2.0, but you have huggingface-hub 0.36.2 which is incompatible.


In [2]:
import pandas as pd
from transformers import Trainer, TrainingArguments, EarlyStoppingCallback, AutoModelForSequenceClassification, AutoTokenizer, set_seed, TrainerCallback
from datasets import Dataset
import joblib
import numpy as np
from google.colab import drive
import os
import json
import zipfile
from sklearn.metrics import f1_score, classification_report
import time
import torch
from transformers.trainer_utils import get_last_checkpoint
from transformers import DataCollatorWithPadding
import adapters
from adapters import DoubleSeqBnConfig, AdapterTrainer
#DoubleBnConfig = Houlsby

#for adapters on ModernBERT
from transformers import AutoModelForSequenceClassification


In [3]:
drive.mount('/content/drive')

SPLIT_PATH = "/content/drive/MyDrive/thesis_results/SCOTBESS_splits/SCOTBESS_FULL_ANNOTATED_TERRA_LOW_SPLIT.csv"
MLB_PATH = "/content/drive/MyDrive/thesis_results/SCOTBESS_splits/scotbess_mlb.joblib"

output_dir = "/content/drive/MyDrive/thesis_results/SCOTBESS_EXPERIMENTS/SCOTBESS_ModernBERT_Houlsby"
os.makedirs(output_dir, exist_ok=True)


Mounted at /content/drive


Loading the dataset

In [4]:
scotbess_df = pd.read_csv(SPLIT_PATH)

In [5]:
scotbess_df_train = scotbess_df[scotbess_df["split"] == "train"].reset_index(drop=True)
scotbess_df_val = scotbess_df[scotbess_df["split"] == "validation"].reset_index(drop=True)
scotbess_df_test = scotbess_df[scotbess_df["split"] == "test"].reset_index(drop=True)

print("Train shape:", scotbess_df_train.shape)
print("Validation shape:", scotbess_df_val.shape)
print("Test shape:", scotbess_df_test.shape)

Train shape: (1340, 7)
Validation shape: (165, 7)
Test shape: (170, 7)


In [6]:
mlb = joblib.load(MLB_PATH)

print("Number of labels:", len(mlb.classes_))
print(mlb.classes_)

Number of labels: 20
['Agricultural Land' 'Community and Economic Benefits'
 'Consultation, Transparency and Information' 'Cumulative Impact'
 'Decommissioning and Site Restoration' 'Emergency Planning and Response'
 'Fire and Explosion Risk' 'Grid Connection and Electrical Infrastructure'
 'Health and Wellbeing' 'Landscape, Visual and Heritage Impact'
 'Light Pollution' 'Noise' 'Planning Policy and Regulatory Compliance'
 'Project Need' 'Property Value'
 'Residential Proximity and Separation Distance' 'Site Selection'
 'Traffic' 'Water and Soil Contamination' 'Wildlife and Ecology']


In [7]:
def parse_labels(value):
    if isinstance(value, list):
        return value
    if pd.isna(value):
        return []
    return json.loads(value)

for df in [scotbess_df_train, scotbess_df_val, scotbess_df_test]:
    df["label_list"] = df["labels"].apply(parse_labels)


In [8]:
scotbess_y_train = mlb.transform(scotbess_df_train["label_list"])
scotbess_y_val = mlb.transform(scotbess_df_val["label_list"])
scotbess_y_test = mlb.transform(scotbess_df_test["label_list"])


In [9]:
scotbess_y_train.shape, scotbess_y_val.shape, scotbess_y_test.shape


((1340, 20), (165, 20), (170, 20))

In [10]:
scotbess_X_train = scotbess_df_train["final_masked_text"].fillna("").astype(str)
scotbess_X_val = scotbess_df_val["final_masked_text"].fillna("").astype(str)
scotbess_X_test = scotbess_df_test["final_masked_text"].fillna("").astype(str)

In [11]:
tokenizer = AutoTokenizer.from_pretrained("answerdotai/ModernBERT-base")

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/694 [00:00<?, ?B/s]

In [12]:
# ModernBERT's token length increased to 8192 (the model's limit) for SCOTBESS
def tokenize(texts):
    return tokenizer(texts.tolist(), truncation=True, max_length=8192)
#dynamic padding
data_collator = DataCollatorWithPadding(tokenizer=tokenizer, pad_to_multiple_of=8)

train_enc = tokenize(scotbess_X_train)
dev_enc = tokenize(scotbess_X_val)
test_enc = tokenize(scotbess_X_test)

In [14]:
y_train_bin = scotbess_y_train.astype(np.float32)
y_dev_bin = scotbess_y_val.astype(np.float32)
y_test_bin = scotbess_y_test.astype(np.float32)

print(mlb.classes_)
print(y_train_bin.shape)


['Agricultural Land' 'Community and Economic Benefits'
 'Consultation, Transparency and Information' 'Cumulative Impact'
 'Decommissioning and Site Restoration' 'Emergency Planning and Response'
 'Fire and Explosion Risk' 'Grid Connection and Electrical Infrastructure'
 'Health and Wellbeing' 'Landscape, Visual and Heritage Impact'
 'Light Pollution' 'Noise' 'Planning Policy and Regulatory Compliance'
 'Project Need' 'Property Value'
 'Residential Proximity and Separation Distance' 'Site Selection'
 'Traffic' 'Water and Soil Contamination' 'Wildlife and Ecology']
(1340, 20)


In [15]:
id2label = {i: label for i, label in enumerate(mlb.classes_)}
label2id = {label: i for i, label in enumerate(mlb.classes_)}

In [16]:
train_dataset = Dataset.from_dict({
    "input_ids": train_enc["input_ids"],
    "attention_mask": train_enc["attention_mask"],
    "labels": y_train_bin.astype("float32")})

eval_dataset = Dataset.from_dict({
    "input_ids": dev_enc["input_ids"],
    "attention_mask": dev_enc["attention_mask"],
    "labels": y_dev_bin.astype("float32")})

test_dataset = Dataset.from_dict({
    "input_ids": test_enc["input_ids"],
    "attention_mask": test_enc["attention_mask"],
    "labels": y_test_bin.astype("float32")})

In [17]:
print(train_dataset[0])
print(len(train_dataset[0]["labels"]))

{'input_ids': [50281, 1231, 452, 4092, 33196, 326, 627, 556, 417, 644, 247, 18731, 22887, 23868, 20023, 1754, 327, 253, 1511, 273, 9378, 281, 320, 908, 275, 253, 4081, 2341, 5718, 9509, 15, 4325, 253, 1491, 12164, 253, 2670, 588, 452, 260, 1884, 35669, 273, 2341, 5718, 407, 247, 2962, 273, 23178, 9378, 5085, 273, 260, 15, 22, 35669, 5350, 15, 3954, 1568, 275, 253, 7177, 1057, 352, 3748, 253, 1511, 273, 9378, 281, 320, 908, 2299, 342, 253, 1655, 4302, 253, 760, 16571, 4500, 651, 320, 27747, 14, 279, 1754, 327, 2341, 4038, 15, 380, 12794, 2495, 273, 2442, 19333, 275, 31976, 1514, 19978, 310, 973, 1929, 285, 973, 14290, 21349, 15, 496, 253, 1982, 835, 9002, 16638, 32560, 27173, 1052, 5718, 9189, 627, 574, 2168, 644, 374, 14, 20, 19333, 15, 844, 2868, 247, 2120, 3907, 2495, 6803, 943, 320, 26237, 1754, 327, 9378, 1511, 313, 284, 359, 476, 760, 5467, 31976, 1514, 428, 42, 251, 10, 347, 247, 9509, 273, 436, 2341, 4038, 1735, 281, 16252, 285, 9787, 3607, 24543, 247, 3289, 2495, 342, 6774, 372

In [18]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred

    probs = 1 / (1 + np.exp(-logits))   # sigmoid
    preds = (probs >= 0.5).astype(int)

    labels = labels.astype(int)

    f1_micro = f1_score(labels, preds, average="micro", zero_division=0)
    f1_macro = f1_score(labels, preds, average="macro", zero_division=0)

    return {
        "f1_micro": f1_micro,
        "f1_macro": f1_macro}

In [19]:
print("Train labels:", y_train_bin.shape)
print("Val labels:", y_dev_bin.shape)
print("Test labels:", y_test_bin.shape)
print("Number of labels:", len(mlb.classes_))

Train labels: (1340, 20)
Val labels: (165, 20)
Test labels: (170, 20)
Number of labels: 20


**Training function**

In [20]:
#helpers for counting times for the final run with ModernBERT, as the run would get disconnected by colab  since it takes a long time to run it with 10 epochs on AAPD; reusing for SCOTBESS
class CheckpointTimeCallback(TrainerCallback):
    def __init__(self, output_dir):
        self.output_dir = output_dir
        self.time_log_path = os.path.join(output_dir, "time_log.json")
        self.previous_time_sec = 0.0
        self.session_start = None
        self.current_total_time_sec = 0.0
        self.current_session_time_sec = 0.0

    def on_train_begin(self, args, state, control, **kwargs):
        if os.path.exists(self.time_log_path):
            with open(self.time_log_path, "r") as f:
                self.previous_time_sec = json.load(f).get("train_time_sec", 0.0)
        else:
            self.previous_time_sec = 0.0

        self.session_start = time.perf_counter()
        self.current_total_time_sec = self.previous_time_sec
        self.current_session_time_sec = 0.0

    def _save_time(self, state):
        if torch.cuda.is_available():
            torch.cuda.synchronize()

        self.current_session_time_sec = time.perf_counter() - self.session_start
        self.current_total_time_sec = self.previous_time_sec + self.current_session_time_sec

        data = {
            "train_time_sec": self.current_total_time_sec,
            "current_session_train_time_sec": self.current_session_time_sec,
            "previous_train_time_sec": self.previous_time_sec,
            "last_global_step": int(state.global_step),
            "last_epoch": float(state.epoch) if state.epoch is not None else None,
        }

        os.makedirs(self.output_dir, exist_ok=True)

        tmp_path = self.time_log_path + ".tmp"
        with open(tmp_path, "w") as f:
            json.dump(data, f, indent=2)

        os.replace(tmp_path, self.time_log_path)

    def on_save(self, args, state, control, **kwargs):
        self._save_time(state)

    def on_train_end(self, args, state, control, **kwargs):
        self._save_time(state)

    def get_times(self):
        if os.path.exists(self.time_log_path):
            with open(self.time_log_path, "r") as f:
                data = json.load(f)

            return (
                data.get("train_time_sec", self.current_total_time_sec),
                data.get("current_session_train_time_sec", self.current_session_time_sec),
            )

        return self.current_total_time_sec, self.current_session_time_sec

In [21]:
def sync_cuda():
    if torch.cuda.is_available():
        torch.cuda.synchronize()

def reset_cuda_peak_memory():
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()
        torch.cuda.synchronize()

#measure vram only for final best config
def get_peak_vram_gb():
    if not torch.cuda.is_available():
        return None

    torch.cuda.synchronize()
    return torch.cuda.max_memory_allocated() / (1024 ** 3)


def run_training(config, seed=0, evaluate_test=False, measure_vram=False, save_report=False):
    set_seed(seed)


    if measure_vram:
        reset_cuda_peak_memory()


    model = AutoModelForSequenceClassification.from_pretrained(config["base_model"],
        num_labels=len(mlb.classes_),
        problem_type="multi_label_classification",
        id2label=id2label,
        label2id=label2id,
        attn_implementation="sdpa")

    adapters.init(model)

    adapter_config = DoubleSeqBnConfig(reduction_factor=config["reduction_factor"])
    model.add_adapter("scotbess", config=adapter_config, set_active=True)
    model.train_adapter("scotbess")

    #Keep the task-specific ModernBERT classification head trainable
    for name, param in model.named_parameters():
       if name.startswith("head.") or name.startswith("classifier."):
            param.requires_grad = True

    #check whether the adapters are working
    print("Active adapters:", model.active_adapters)
    print(model.adapter_summary())
    #check the classification head
    print("Trainable classification/head parameters:")
    for name, param in model.named_parameters():
        if param.requires_grad and ("head" in name.lower() or "classifier" in name.lower() or "classification" in name.lower()):
           print(name, param.numel())


    #for the final statistics
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total_params = sum(p.numel() for p in model.parameters())

    training_args = TrainingArguments(
        output_dir=config["output_dir"],

        learning_rate=config["learning_rate"],
        per_device_train_batch_size=config["micro_batch_size"],
        per_device_eval_batch_size=config["micro_batch_size"],
        gradient_accumulation_steps=config["gradient_accumulation_steps"],

        num_train_epochs=config["num_train_epochs"],
        weight_decay=config["weight_decay"],
        warmup_ratio=config["warmup_ratio"],

        eval_strategy="epoch",
        save_strategy="epoch",
        logging_strategy="epoch",

        load_best_model_at_end=True,
        metric_for_best_model="eval_f1_macro",
        greater_is_better=True,

        save_total_limit=2,
        fp16=True,
        report_to="none",
        #gradient checkpointing
        gradient_checkpointing=True,
        gradient_checkpointing_kwargs={"use_reentrant": False})

    time_callback = CheckpointTimeCallback(config["output_dir"])

    #adapter trainer, as recommended in the library's documentation
    trainer = AdapterTrainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=eval_dataset,
        data_collator=data_collator,
        compute_metrics=compute_metrics,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=config["early_stopping_patience"]), time_callback])


    #check
    print("Active adapters after trainer creation:", trainer.model.active_adapters)

#for resuming if something goes wrong
    last_checkpoint = None
    if os.path.isdir(config["output_dir"]):
        last_checkpoint = get_last_checkpoint(config["output_dir"])

    if last_checkpoint is not None:
        print(f"Resuming from checkpoint: {last_checkpoint}")
    else:
        print("Starting training from scratch.")


####imporved for the final run on MODERNBERT
    sync_cuda()
    # includes training + epoch validation + checkpoint saving + early stopping + loading best model
    trainer.train(resume_from_checkpoint=last_checkpoint)
    sync_cuda()
    # accumulated training time saved after completed checkpoints/epochs
    train_time_sec, current_session_train_time_sec = time_callback.get_times()


    if measure_vram:
        training_peak_vram_gb = get_peak_vram_gb()
    else:
        training_peak_vram_gb = None

    sync_cuda()
    val_start = time.perf_counter()

    val_results = trainer.evaluate(eval_dataset, metric_key_prefix="val")

    sync_cuda()
    val_eval_time_sec = time.perf_counter() - val_start

    result = {
        "model": config["base_model"],
        "dataset": "Scot-BESS",
        "method": "houlsby",
        "seed": seed,

        "learning_rate": config["learning_rate"],
        "micro_batch_size": config["micro_batch_size"],
        "gradient_accumulation_steps": config["gradient_accumulation_steps"],
        "effective_batch_size": config["effective_batch_size"],
        "num_train_epochs": config["num_train_epochs"],

        "best_checkpoint": trainer.state.best_model_checkpoint,
        "best_metric": trainer.state.best_metric,
        "actual_epochs_trained": trainer.state.epoch,

        "train_time_sec": train_time_sec,
        "current_session_train_time_sec": current_session_train_time_sec,
        "val_eval_time_sec": val_eval_time_sec,

        "training_peak_vram_gb": training_peak_vram_gb,

        "trainable_params": trainable_params,
        "total_params": total_params,

        "val_f1_macro": val_results["val_f1_macro"],
        "val_f1_micro": val_results["val_f1_micro"]}

    if evaluate_test:
        sync_cuda()
        test_start = time.perf_counter()

        # single forward pass — gives metrics + raw predictions
        test_pred_output = trainer.predict(test_dataset)

        sync_cuda()
        test_eval_time_sec = time.perf_counter() - test_start

        # derive predictions - needed for classification report
        test_probs = 1 / (1 + np.exp(-test_pred_output.predictions))
        test_binary_preds = (test_probs >= 0.5).astype(int)
        gold_labels = (test_pred_output.label_ids >= 0.5).astype(int)

        test_metrics = test_pred_output.metrics

        result.update({
            "test_eval_time_sec": test_eval_time_sec,
            "test_inference_per_sample_ms": (test_eval_time_sec / len(test_dataset)) * 1000,
            "test_f1_macro": test_metrics["test_f1_macro"],
            "test_f1_micro": test_metrics["test_f1_micro"],
            "avg_predicted_labels": float(test_binary_preds.sum(axis=1).mean()),
            "avg_gold_labels": float(gold_labels.sum(axis=1).mean()),})

        if save_report:
            report_dict = classification_report(
                gold_labels, test_binary_preds,
                target_names=mlb.classes_, zero_division=0, output_dict=True)
            report_df = pd.DataFrame(report_dict).T
            report_path = os.path.join(output_dir, f"classification_report_seed_{seed}.csv")
            report_df.to_csv(report_path)
            print(f"Classification report saved to {report_path}")

            #saving raw arrays for possible future analysis
            predictions_path = os.path.join(
                output_dir,
                f"test_predictions_seed_{seed}.npz")
            np.savez_compressed(
                predictions_path,
                y_true=gold_labels,
                y_pred=test_binary_preds,
                y_prob=test_probs,
                label_names=np.array(mlb.classes_),
                threshold=np.array([0.5]))
            result["test_predictions_path"] = predictions_path
            print(f"Predictions saved to {predictions_path}")

            #for saving the adapter
            adapter_save_path = os.path.join(config["output_dir"], "final_Houlsby_adapter")
            trainer.model.save_adapter(adapter_save_path, "scotbess")
            result["saved_adapter_path"] = adapter_save_path
            print(f"Adapter saved to {adapter_save_path}")

            #for saving ModernBERt classification head
            head_save_path = os.path.join(config["output_dir"], "modernbert_classification_head.pt")
            torch.save({ "head": trainer.model.head.state_dict(), "classifier": trainer.model.classifier.state_dict(),}, head_save_path)


    result["total_measured_time_sec"] = (result["train_time_sec"] + result["val_eval_time_sec"] + result.get("test_eval_time_sec", 0))

    return result

In [22]:
#fixed params
base_config = {
    "output_dir": os.path.join(output_dir, "search"),
    "base_model": "answerdotai/ModernBERT-base",
    "tokenizer_name": "answerdotai/ModernBERT-base",

    "max_length": 8192,
    "num_train_epochs": 4, #fewer epochs for ModernBERT, for search only
    "weight_decay": 0.01,
    "warmup_ratio": 0.1,
    "early_stopping_patience": 3,
    "micro_batch_size": 4,

    "reduction_factor": 8}  #the default one is 16, but I will use 8 (same as the one used for DistilBERT), this is also the choice of Razuvayevskaya et al. (2024)


learning_rates = [1e-4, 2e-4, 5e-4] #1e-4 is recommmended in the adapters library documentation, 2e-4 is used by Razuvayevskaya et al. (2024)
effective_batch_sizes = [8, 16]


search_results_path = os.path.join(output_dir, "search_results.csv")
search_results = []

for lr in learning_rates:
    for effective_bs in effective_batch_sizes:
        config = base_config.copy()
        config["learning_rate"] = lr
        config["effective_batch_size"] = effective_bs
        config["gradient_accumulation_steps"] = (effective_bs // config["micro_batch_size"])
        config["output_dir"] = (f"{base_config['output_dir']}/lr_{lr}_efbs_{effective_bs}")

        #skipping already-completed configs on resume
        if os.path.exists(search_results_path):
            existing = pd.read_csv(search_results_path)
            already_done = existing[
                (existing["learning_rate"] == lr) &
                (existing["effective_batch_size"] == effective_bs)]
            if len(already_done) > 0:
                print(f"Skipping lr={lr}, effective_bs={effective_bs} (already done)")
                search_results.append(already_done.iloc[0].to_dict())
                continue

        print("=" * 80)
        print(f"Running ModernBERT: lr={lr},  effective_batch_size={effective_bs}, grad_accum={config['gradient_accumulation_steps']}")
        print("=" * 80)

        result = run_training(config, seed=0)
        search_results.append(result)

        #saving incrementally after every config
        pd.DataFrame(search_results).to_csv(search_results_path, index=False)

search_results_df = pd.DataFrame(search_results)
search_results_df = search_results_df.sort_values("val_f1_macro", ascending=False).reset_index(drop=True)
search_results_df.to_csv(search_results_path, index=False)
#!!! train_time_sec in search results is unreliable due to checkpoint resumption !!!
# !!!timing is only reported from the final seed runs - I ensured the run is not resumed
search_results_df

Running ModernBERT: lr=0.0001,  effective_batch_size=8, grad_accum=2


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/599M [00:00<?, ?B/s]

Some weights of ModernBertForSequenceClassification were not initialized from the model checkpoint at answerdotai/ModernBERT-base and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Active adapters: Stack[scotbess]
Name                     Architecture         #Param      %Param  Active   Train
--------------------------------------------------------------------------------
scotbess                 bottleneck        6,526,080       4.378       1       1
--------------------------------------------------------------------------------
Full model                               149,064,960     100.000               0
Trainable classification/head parameters:
head.dense.weight 589824
head.norm.weight 768
classifier.weight 15360
classifier.bias 20
Active adapters after trainer creation: Stack[scotbess]
Starting training from scratch.


...

Some weights of ModernBertForSequenceClassification were not initialized from the model checkpoint at answerdotai/ModernBERT-base and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Active adapters: Stack[scotbess]
Name                     Architecture         #Param      %Param  Active   Train
--------------------------------------------------------------------------------
scotbess                 bottleneck        6,526,080       4.378       1       1
--------------------------------------------------------------------------------
Full model                               149,064,960     100.000               0
Trainable classification/head parameters:
head.dense.weight 589824
head.norm.weight 768
classifier.weight 15360
classifier.bias 20
Active adapters after trainer creation: Stack[scotbess]
Starting training from scratch.


Epoch,Training Loss,Validation Loss,F1 Micro,F1 Macro
1,1.927700,0.392754,0.662464,0.527113
2,1.292700,0.290346,0.797290,0.650908
3,0.930100,0.257025,0.828378,0.756128
4,0.662200,0.233431,0.852443,0.807099


early stopping required metric_for_best_model, but did not find eval_f1_macro so early stopping is disabled


,model,dataset,method,seed,learning_rate,micro_batch_size,gradient_accumulation_steps,effective_batch_size,num_train_epochs,best_checkpoint,...,actual_epochs_trained,train_time_sec,current_session_train_time_sec,val_eval_time_sec,training_peak_vram_gb,trainable_params,total_params,val_f1_macro,val_f1_micro,total_measured_time_sec
0,answerdotai/ModernBERT-base,Scot-BESS,houlsby,0,0.0005,4,2,8,4,/content/drive/MyDrive/thesis_results/SCOTBESS...,...,4.0,1757.391759,1757.391759,13.807587,None,7132052,156197012,0.833992,0.869653,1771.199345
1,answerdotai/ModernBERT-base,Scot-BESS,houlsby,0,0.0005,4,4,16,4,/content/drive/MyDrive/thesis_results/SCOTBESS...,...,4.0,1746.223908,1746.223908,13.713556,None,7132052,156197012,0.807099,0.852443,1759.937464
2,answerdotai/ModernBERT-base,Scot-BESS,houlsby,0,0.0002,4,2,8,4,/content/drive/MyDrive/thesis_results/SCOTBESS...,...,4.0,1727.943916,1727.943916,13.601250,None,7132052,156197012,0.777902,0.832747,1741.545166
3,answerdotai/ModernBERT-base,Scot-BESS,houlsby,0,0.0002,4,4,16,4,/content/drive/MyDrive/thesis_results/SCOTBESS...,...,4.0,1761.635781,1761.635781,13.856881,None,7132052,156197012,0.719425,0.806911,1775.492661
4,answerdotai/ModernBERT-base,Scot-BESS,houlsby,0,0.0001,4,2,8,4,/content/drive/MyDrive/thesis_results/SCOTBESS...,...,4.0,1801.547102,1801.547102,13.812830,None,7132052,156197012,0.685325,0.780315,1815.359932
5,answerdotai/ModernBERT-base,Scot-BESS,houlsby,0,0.0001,4,4,16,4,/content/drive/MyDrive/thesis_results/SCOTBESS...,...,4.0,1727.814147,1727.814147,13.583949,None,7132052,156197012,0.651087,0.739530,1741.398096


In [23]:
best_row = search_results_df.iloc[0]

best_lr = float(best_row["learning_rate"])
best_effective_batch_size = int(best_row["effective_batch_size"])
best_grad_accum = int(best_row["gradient_accumulation_steps"])

print("Best learning rate:", best_lr)
print("Best effective batch size:", best_effective_batch_size)
print("Gradient accumulation steps:", best_grad_accum)
print("Best validation macro-F1:", best_row["val_f1_macro"])
print("Best checkpoint:", best_row["best_checkpoint"])

Best learning rate: 0.0005
Best effective batch size: 8
Gradient accumulation steps: 2
Best validation macro-F1: 0.8339916773243182
Best checkpoint: /content/drive/MyDrive/thesis_results/SCOTBESS_EXPERIMENTS/SCOTBESS_ModernBERT_Houlsby/search/lr_0.0005_efbs_8/checkpoint-672


In [24]:
best_config = base_config.copy()
best_config["learning_rate"] = best_lr
best_config["effective_batch_size"] = best_effective_batch_size
best_config["gradient_accumulation_steps"] = best_grad_accum
best_config["selection_metric"] = "val_f1_macro"
best_config["best_validation_macro_f1"] = float(best_row["val_f1_macro"])
best_config["best_validation_micro_f1"] = float(best_row["val_f1_micro"])
best_config["best_checkpoint_from_search"] = best_row["best_checkpoint"]

best_config_path = os.path.join(output_dir, "best_config.json")

with open(best_config_path, "w") as f:
    json.dump(best_config, f, indent=2)


**Final run (test set) on the best found configuration**

In [25]:
with open(best_config_path, "r") as f:
    final_config = json.load(f)

#final runs use the full training budget with early stopping
final_config["num_train_epochs"] = 10

In [26]:
test_output_dir = os.path.join(output_dir, "test")
os.makedirs(test_output_dir, exist_ok=True)

test_results_path = os.path.join(test_output_dir, "SCOTBESS_ModernBERT__Houlsby_test_results.csv")
test_results = []

for seed in [0, 1, 2]:
    config = final_config.copy()
    config["seed"] = seed
    config["output_dir"] = (os.path.join(test_output_dir, f"SCOTBESS_ModernBERT_Houlsby_test_seed_{seed}"))

    # Skip already-completed seeds on resume
    if os.path.exists(test_results_path):
        existing = pd.read_csv(test_results_path)
        already_done = existing[existing["seed"] == seed]
        if len(already_done) > 0:
            print(f"Skipping seed={seed} (already done)")
            test_results.append(already_done.iloc[0].to_dict())
            continue

    print("=" * 80)
    print(f"Final run: seed={seed}, lr={final_config['learning_rate']}, micro_batch={final_config['micro_batch_size']}, effective_batch={final_config['effective_batch_size']}, grad_accum={final_config['gradient_accumulation_steps']}")
    print("=" * 80)
    result = run_training(config, seed=seed, evaluate_test=True, measure_vram=True, save_report = True)
    test_results.append(result)

    # Save incrementally after every seed
    pd.DataFrame(test_results).to_csv(test_results_path, index=False)

test_results_df = pd.DataFrame(test_results)
test_results_df.to_csv(test_results_path, index=False)
test_results_df

Final run: seed=0, lr=0.0005, micro_batch=4, effective_batch=8, grad_accum=2


Some weights of ModernBertForSequenceClassification were not initialized from the model checkpoint at answerdotai/ModernBERT-base and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Active adapters: Stack[scotbess]
Name                     Architecture         #Param      %Param  Active   Train
--------------------------------------------------------------------------------
scotbess                 bottleneck        6,526,080       4.378       1       1
--------------------------------------------------------------------------------
Full model                               149,064,960     100.000               0
Trainable classification/head parameters:
head.dense.weight 589824
head.norm.weight 768
classifier.weight 15360
classifier.bias 20
Active adapters after trainer creation: Stack[scotbess]
Starting training from scratch.


Epoch,Training Loss,Validation Loss,F1 Micro,F1 Macro
1,0.961600,0.412898,0.636740,0.481095
2,0.623100,0.284564,0.798383,0.689780
3,0.418100,0.246333,0.830169,0.785342
4,0.278000,0.229805,0.859615,0.845872
5,0.169800,0.241954,0.873984,0.842587
6,0.096200,0.258683,0.875310,0.860070
7,0.047100,0.305070,0.868380,0.847724
8,0.019600,0.259567,0.892717,0.883578
9,0.007500,0.263620,0.893300,0.880462
10,0.004100,0.264689,0.892695,0.878698


early stopping required metric_for_best_model, but did not find eval_f1_macro so early stopping is disabled


...

Active adapters: Stack[scotbess]
Name                     Architecture         #Param      %Param  Active   Train
--------------------------------------------------------------------------------
scotbess                 bottleneck        6,526,080       4.378       1       1
--------------------------------------------------------------------------------
Full model                               149,064,960     100.000               0
Trainable classification/head parameters:
head.dense.weight 589824
head.norm.weight 768
classifier.weight 15360
classifier.bias 20
Active adapters after trainer creation: Stack[scotbess]
Starting training from scratch.


Epoch,Training Loss,Validation Loss,F1 Micro,F1 Macro
1,0.970600,0.403958,0.685365,0.595222
2,0.619400,0.278661,0.803653,0.698826
3,0.412700,0.225128,0.848214,0.801051
4,0.256300,0.228159,0.861656,0.851521
5,0.151600,0.225685,0.885473,0.866221
6,0.082600,0.226734,0.889442,0.879671
7,0.039900,0.233712,0.895418,0.884598
8,0.014300,0.235992,0.901903,0.895550
9,0.005600,0.230725,0.907186,0.900074
10,0.003500,0.231220,0.907731,0.901469


early stopping required metric_for_best_model, but did not find eval_f1_macro so early stopping is disabled


Classification report saved to /content/drive/MyDrive/thesis_results/SCOTBESS_EXPERIMENTS/SCOTBESS_ModernBERT_Houlsby/classification_report_seed_2.csv
Predictions saved to /content/drive/MyDrive/thesis_results/SCOTBESS_EXPERIMENTS/SCOTBESS_ModernBERT_Houlsby/test_predictions_seed_2.npz
Adapter saved to /content/drive/MyDrive/thesis_results/SCOTBESS_EXPERIMENTS/SCOTBESS_ModernBERT_Houlsby/test/SCOTBESS_ModernBERT_Houlsby_test_seed_2/final_Houlsby_adapter


,model,dataset,method,seed,learning_rate,micro_batch_size,gradient_accumulation_steps,effective_batch_size,num_train_epochs,best_checkpoint,...,val_f1_micro,test_eval_time_sec,test_inference_per_sample_ms,test_f1_macro,test_f1_micro,avg_predicted_labels,avg_gold_labels,test_predictions_path,saved_adapter_path,total_measured_time_sec
0,answerdotai/ModernBERT-base,Scot-BESS,houlsby,0,0.0005,4,2,8,10,/content/drive/MyDrive/thesis_results/SCOTBESS...,...,0.892717,14.007683,82.398137,0.873280,0.890518,6.117647,5.917647,/content/drive/MyDrive/thesis_results/SCOTBESS...,/content/drive/MyDrive/thesis_results/SCOTBESS...,4409.641870
1,answerdotai/ModernBERT-base,Scot-BESS,houlsby,1,0.0005,4,2,8,10,/content/drive/MyDrive/thesis_results/SCOTBESS...,...,0.892000,13.851448,81.479104,0.858977,0.881070,5.952941,5.917647,/content/drive/MyDrive/thesis_results/SCOTBESS...,/content/drive/MyDrive/thesis_results/SCOTBESS...,4411.679106
2,answerdotai/ModernBERT-base,Scot-BESS,houlsby,2,0.0005,4,2,8,10,/content/drive/MyDrive/thesis_results/SCOTBESS...,...,0.907731,13.866063,81.565077,0.880137,0.897000,5.847059,5.917647,/content/drive/MyDrive/thesis_results/SCOTBESS...,/content/drive/MyDrive/thesis_results/SCOTBESS...,4424.558978


In [27]:
test_summary_df = test_results_df[[
    "test_f1_macro",
    "test_f1_micro",
    "avg_predicted_labels",
    "avg_gold_labels",
    "train_time_sec",
    "val_eval_time_sec",
    "test_eval_time_sec",
    "test_inference_per_sample_ms",
    "training_peak_vram_gb",
    "actual_epochs_trained",
    "trainable_params",
    "total_params",
    "total_measured_time_sec"]].agg(["mean", "std"])

test_summary_path =  os.path.join(test_output_dir, "SCOTBESS_ModernBERT_Houlsby_test_results_summary.csv")
test_summary_df.to_csv(test_summary_path)

test_summary_df

,test_f1_macro,test_f1_micro,avg_predicted_labels,avg_gold_labels,train_time_sec,val_eval_time_sec,test_eval_time_sec,test_inference_per_sample_ms,training_peak_vram_gb,actual_epochs_trained,trainable_params,total_params,total_measured_time_sec
mean,0.870798,0.889529,5.972549,5.917647,4387.693027,13.691893,13.908398,81.814106,7.254921,10.0,7132052.0,156197012.0,4415.293318
std,0.010796,0.008011,0.136356,0.000000,8.164806,0.110325,0.086294,0.507609,0.000809,0.0,0.0,0.0,8.088691


In [28]:
from google.colab import runtime
runtime.unassign()